# Analysis Notebook
This notebook implements Plan 08 in a code-first, reproducible way for non-data-scientist users.

Run order:
1. Run `python src/cleaning.py` first.
2. Open this notebook and use **Run All**.
3. Review outputs in `outputs/tables`, `outputs/figures`, and `outputs/analysis_report.html`.

## What Each Section Does
1. Configuration: defines file paths, station settings, and modeling toggles.
2. Environment setup: imports libraries and prepares output folders.
3. Guardrails/helpers: validates schema and provides shared metrics.
4. Data loading: reads cleaned hourly data and enforces timestamp integrity.
5. Data quality: calculates coverage, missingness, and noon-readiness.
6. Similarity EDA: compares station pairs across core variables and seasons.
7. FWI modeling: computes daily moisture codes and FWI, then validates against ECCC.
8. Redundancy modeling: uses PCA, clustering, and benchmarking vs Stanhope.
9. Decision layer: estimates uncertainty and exports a plain-language report.

In [7]:
# =============================================================================
# CONFIGURATION CELL
# Purpose: centralize all run-time settings in one place.
# Junior note: change values here instead of editing later logic cells.
# =============================================================================

from pathlib import Path

# Base project paths (assumes notebook launched from project root).
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
SCRUBBED_DIR = PROJECT_ROOT / "data" / "scrubbed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
LOGS_DIR = OUTPUTS_DIR / "logs"

# Core inputs used by this notebook.
HOURLY_INPUT = SCRUBBED_DIR / "02_hourly_weather_utc.csv"
REFERENCE_FWI_DIR = RAW_DIR / "ECCC_Stanhope_FWI"

# Business comparison targets.
REFERENCE_STATION = "Stanhope"
CANDIDATE_STATIONS = ["Cavendish", "Greenwich"]
FWI_DELTA_THRESHOLD = 2.0

# Reproducibility and model controls.
RANDOM_SEED = 42
START_DATE_UTC = None  # Optional filter, e.g. "2023-01-01"
END_DATE_UTC = None    # Optional filter, e.g. "2025-12-31"
DO_KMEANS = True
K_VALUES = [2, 3, 4]
PLOTTING_STYLE = "whitegrid"

**Expected Output (Cell 2)**
- No table output is expected.
- This cell should run silently with no errors.
- Key variables (paths, thresholds, station names) are now available for all downstream cells.

In [8]:
# =============================================================================
# ENVIRONMENT SETUP CELL
# Purpose: import libraries, set plotting style, and ensure output directories.
# =============================================================================

import math
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Make random operations deterministic for reproducible results.
np.random.seed(RANDOM_SEED)
sns.set_theme(style=PLOTTING_STYLE)

# Ensure all output folders exist before writing any artifacts.
for path in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Run All note: This notebook writes production artifacts to outputs/tables and outputs/figures.")
print(f"Project root: {PROJECT_ROOT}")
display(Markdown("### Business-Friendly Output Mode\nThis notebook now both saves artifact files and shows the most important summaries directly below each section."))

Run All note: This notebook writes production artifacts to outputs/tables and outputs/figures.
Project root: c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC


### Business-Friendly Output Mode
This notebook now both saves artifact files and shows the most important summaries directly below each section.

**Expected Output (Cell 3)**
- Prints project/run messages, including the detected project root.
- Creates output directories if they do not already exist.
- Displays a short markdown note confirming business-friendly output mode.

In [9]:
# =============================================================================
# GUARDRAIL AND METRIC HELPER CELL
# Purpose: define validation helpers and shared metric functions used later.
# =============================================================================

HALIFAX_TZ = ZoneInfo("America/Halifax")
UTC_TZ = ZoneInfo("UTC")

# Canonical variables expected from cleaned hourly data.
CANONICAL_VARS = [
    "air_temperature_c",
    "relative_humidity_pct",
    "wind_speed_kmh",
    "wind_direction_deg",
    "precipitation_mm",
]
CORE_VARS = ["air_temperature_c", "relative_humidity_pct", "wind_speed_kmh"]


def fail(msg: str) -> None:
    """Raise a hard-stop error with a readable message."""
    raise ValueError(msg)


def require_columns(df: pd.DataFrame, required: list[str], name: str) -> None:
    """Fail early if required columns are missing from an input dataframe."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        fail(f"{name} missing required columns: {missing}")


def to_numeric_safe(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert selected columns to numeric and coerce non-numeric values to NaN."""
    out = df.copy()
    for col in cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def calc_rmse(a: pd.Series, b: pd.Series) -> float:
    """Root Mean Squared Error between aligned series."""
    if len(a) == 0:
        return float("nan")
    return float(np.sqrt(np.nanmean((a - b) ** 2)))


def calc_mae(a: pd.Series, b: pd.Series) -> float:
    """Mean Absolute Error between aligned series."""
    if len(a) == 0:
        return float("nan")
    return float(np.nanmean(np.abs(a - b)))


def calc_bias(a: pd.Series, b: pd.Series) -> float:
    """Signed mean error (positive means first series is higher on average)."""
    if len(a) == 0:
        return float("nan")
    return float(np.nanmean(a - b))


def calc_corr(a: pd.Series, b: pd.Series, method: str = "pearson") -> float:
    """Correlation helper with minimum length protection."""
    if len(a) < 3:
        return float("nan")
    return float(a.corr(b, method=method))


print("Guardrails initialized.")

Guardrails initialized.


**Expected Output (Cell 4)**
- Prints `Guardrails initialized.`
- No data tables are created yet.
- Helper functions and metric utilities are now defined and ready for use.

In [10]:
# =============================================================================
# LOAD CLEANED HOURLY DATA
# Purpose: read scrubbed data and enforce strict structural/time integrity.
# =============================================================================

# Hard-stop if the required cleaned file is not present.
if not HOURLY_INPUT.exists():
    fail(f"Missing cleaned input. Run cleaning first: {HOURLY_INPUT}")

# Load full hourly dataset produced by src/cleaning.py.
hourly = pd.read_csv(HOURLY_INPUT, low_memory=False)
required_cols = ["datetime_utc", "station_slug", "station_raw", "source", *CANONICAL_VARS]
require_columns(hourly, required_cols, "hourly")

# Coerce weather variables to numeric (bad values become NaN).
hourly = to_numeric_safe(hourly, CANONICAL_VARS)

# Parse and validate UTC timestamps.
hourly["datetime_utc"] = pd.to_datetime(hourly["datetime_utc"], utc=True, errors="coerce")
if hourly["datetime_utc"].isna().any():
    fail(f"Invalid UTC timestamps found: {int(hourly['datetime_utc'].isna().sum())}")

# Optional user-supplied date filters.
if START_DATE_UTC:
    hourly = hourly[hourly["datetime_utc"] >= pd.Timestamp(START_DATE_UTC, tz="UTC")]
if END_DATE_UTC:
    hourly = hourly[hourly["datetime_utc"] <= pd.Timestamp(END_DATE_UTC, tz="UTC")]

# Ensure one row per station per hour.
dupes = hourly.duplicated(subset=["station_slug", "datetime_utc"])
if bool(dupes.any()):
    fail(f"Duplicate station/timestamp rows found: {int(dupes.sum())}")

# Keep rows sorted for deterministic downstream behavior.
hourly = hourly.sort_values(["station_slug", "datetime_utc"]).reset_index(drop=True)
print(f"Loaded rows: {len(hourly):,}")
print(f"Stations: {hourly['station_slug'].nunique()}")
print(f"UTC range: {hourly['datetime_utc'].min()} -> {hourly['datetime_utc'].max()}")

Loaded rows: 166,087
Stations: 6
UTC range: 2022-01-01 04:00:00+00:00 -> 2026-01-01 03:00:00+00:00


**Expected Output (Cell 5)**
- Prints row count, number of stations, and UTC date range.
- If data issues exist (missing file, bad timestamps, duplicates), the cell stops with a clear error.
- On success, the clean `hourly` dataframe is ready for quality and modeling steps.

In [11]:
# =============================================================================
# DATA QUALITY ANALYSIS
# Purpose: quantify coverage, missingness, and noon-readiness for FWI workflows.
# =============================================================================

# ---- Overall coverage by station and variable ----
coverage_rows = []
for (station_slug, station_raw), g in hourly.groupby(["station_slug", "station_raw"], dropna=False):
    total_hours = len(g)
    for var in CANONICAL_VARS:
        missing_hours = int(g[var].isna().sum())
        coverage_rows.append(
            {
                "station_slug": station_slug,
                "station_raw": station_raw,
                "variable": var,
                "total_hours": total_hours,
                "missing_hours": missing_hours,
                "missing_pct": (missing_hours / total_hours * 100.0) if total_hours else np.nan,
            }
        )
coverage_overall = pd.DataFrame(coverage_rows)
coverage_overall.to_csv(TABLES_DIR / "03_explore_coverage_overall.csv", index=False)

# ---- Monthly coverage trends (helps detect season-specific outages) ----
work = hourly.copy()
work["year_month"] = work["datetime_utc"].dt.tz_localize(None).dt.to_period("M").astype(str)
monthly_rows = []
for (station_slug, station_raw, ym), g in work.groupby(["station_slug", "station_raw", "year_month"], dropna=False):
    total_hours = len(g)
    for var in CANONICAL_VARS:
        missing_hours = int(g[var].isna().sum())
        monthly_rows.append(
            {
                "station_slug": station_slug,
                "station_raw": station_raw,
                "year_month": ym,
                "variable": var,
                "total_hours": total_hours,
                "missing_hours": missing_hours,
                "missing_pct": (missing_hours / total_hours * 100.0) if total_hours else np.nan,
            }
        )
coverage_monthly = pd.DataFrame(monthly_rows)
coverage_monthly.to_csv(TABLES_DIR / "03_explore_coverage_monthly.csv", index=False)

# ---- Noon readiness for fire-weather operations ----
# FWI workflows require reliable noon meteorology and a complete prior 24h precip window.
noon_rows = []
for station_slug, g in hourly.groupby("station_slug", dropna=False):
    s = g.copy()
    s["datetime_local"] = s["datetime_utc"].dt.tz_convert(HALIFAX_TZ)
    s["local_date"] = s["datetime_local"].dt.date
    s["local_hour"] = s["datetime_local"].dt.hour
    noon = s[s["local_hour"] == 12].copy()
    days_with_noon = int(noon["local_date"].nunique())
    core_ready = noon[CORE_VARS].notna().all(axis=1)
    days_core_ready = int(noon.loc[core_ready, "local_date"].nunique())

    precip = s.set_index("datetime_utc")["precipitation_mm"]
    precip_ready = 0
    for d in sorted(noon["local_date"].dropna().unique().tolist()):
        # Rain window: previous local day 13:00 through current local day 12:00 (24 hourly values).
        start_local = datetime.combine(d - timedelta(days=1), datetime.min.time().replace(hour=13), tzinfo=HALIFAX_TZ)
        window_local = pd.date_range(start=start_local, periods=24, freq="h", tz=HALIFAX_TZ)
        vals = precip.reindex(window_local.tz_convert(timezone.utc))
        if int(vals.notna().sum()) == 24:
            precip_ready += 1

    noon_rows.append(
        {
            "station_slug": station_slug,
            "days_with_noon_row": days_with_noon,
            "days_noon_ready_core": days_core_ready,
            "days_noon_core_not_ready": days_with_noon - days_core_ready,
            "days_precip_ready_24h": precip_ready,
            "noon_core_ready_pct": (days_core_ready / days_with_noon * 100.0) if days_with_noon else np.nan,
            "precip_ready_24h_pct": (precip_ready / days_with_noon * 100.0) if days_with_noon else np.nan,
        }
    )

noon_readiness = pd.DataFrame(noon_rows)
noon_readiness.to_csv(TABLES_DIR / "03_explore_noon_readiness.csv", index=False)

# ---- Business-facing summaries shown inline ----
display(Markdown("### Data Quality Summary (Business Impact)"))
display(Markdown("These numbers show whether each station is reliable enough for fire-risk decisions. Lower missingness and higher noon readiness means more trustworthy daily risk reporting."))

quality_summary = (
    coverage_overall[coverage_overall["variable"].isin(CORE_VARS)]
    .groupby("station_slug", dropna=False)["missing_pct"]
    .mean()
    .reset_index(name="avg_missing_pct_core_vars")
    .merge(
        noon_readiness[["station_slug", "noon_core_ready_pct", "precip_ready_24h_pct"]],
        on="station_slug",
        how="left",
    )
    .sort_values(["avg_missing_pct_core_vars", "noon_core_ready_pct"], ascending=[True, False])
    .reset_index(drop=True)
    .rename(columns={
        "station_slug": "Station",
        "avg_missing_pct_core_vars": "Avg Missing % (Core Weather)",
        "noon_core_ready_pct": "Noon Core Ready %",
        "precip_ready_24h_pct": "24h Rain Window Ready %",
    })
)
display(quality_summary.round(2))

lowest_quality = quality_summary.tail(1)
if not lowest_quality.empty:
    lq = lowest_quality.iloc[0]
    display(Markdown(
        f"**Priority Data-Quality Risk:** {lq['Station']} has the weakest readiness profile in this run. "
        "If this station is retained for operational decisions, consider targeted sensor checks or redundancy backup."
    ))

print("Saved quality tables.")

### Data Quality Summary (Business Impact)

These numbers show whether each station is reliable enough for fire-risk decisions. Lower missingness and higher noon readiness means more trustworthy daily risk reporting.

,Station,Avg Missing % (Core Weather),Noon Core Ready %,24h Rain Window Ready %
0,Stanhope,1.57,98.08,95.62
1,North_Rustico_Wharf,2.94,94.46,99.80
2,Cavendish,7.07,92.74,92.56
3,Greenwich,22.61,51.80,92.01
4,Tracadie_Wharf,39.08,0.00,92.04
5,Stanley_Bridge_Wharf,57.99,0.00,69.94


**Priority Data-Quality Risk:** Stanley_Bridge_Wharf has the weakest readiness profile in this run. If this station is retained for operational decisions, consider targeted sensor checks or redundancy backup.

Saved quality tables.


**Expected Output (Cell 6)**
- Saves data-quality CSVs to `outputs/tables` (coverage overall/monthly + noon readiness).
- Displays a business-facing quality summary table in the notebook.
- Prints `Saved quality tables.` at completion.

In [12]:
# =============================================================================
# EDA + PAIRWISE STATION SIMILARITY
# Purpose: measure interchangeability and micro-climate differences between stations.
# =============================================================================

stations = sorted(hourly["station_slug"].dropna().unique().tolist())
pairs = [(a, b) for i, a in enumerate(stations) for b in stations[i + 1 :]]

# Output collectors.
core_rows = []
precip_rows = []
seasonal_rows = []
wind_uv_rows = []

work = hourly.copy()
work["season"] = work["datetime_utc"].dt.month.map({12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring", 6: "summer", 7: "summer", 8: "summer", 9: "fall", 10: "fall", 11: "fall"})

for a, b in pairs:
    # Inner-join by timestamp so all pairwise metrics are computed on aligned hours only.
    da = work[work["station_slug"] == a][["datetime_utc", *CANONICAL_VARS, "season"]].rename(columns={c: f"{c}_a" for c in CANONICAL_VARS + ["season"]})
    db = work[work["station_slug"] == b][["datetime_utc", *CANONICAL_VARS, "season"]].rename(columns={c: f"{c}_b" for c in CANONICAL_VARS + ["season"]})
    m = da.merge(db, on="datetime_utc", how="inner")

    # Core weather metrics (temp, humidity, wind speed).
    for var in CORE_VARS:
        xa = m[f"{var}_a"]
        xb = m[f"{var}_b"]
        valid = xa.notna() & xb.notna()
        vv_a = xa[valid]
        vv_b = xb[valid]
        core_rows.append(
            {
                "station_a": a,
                "station_b": b,
                "variable": var,
                "aligned_hours": int(valid.sum()),
                "mae": calc_mae(vv_a, vv_b),
                "rmse": calc_rmse(vv_a, vv_b),
                "bias": calc_bias(vv_a, vv_b),
                "pearson_r": calc_corr(vv_a, vv_b, "pearson"),
                "spearman_r": calc_corr(vv_a, vv_b, "spearman"),
            }
        )

    # Precipitation comparison tracked separately.
    xa = m["precipitation_mm_a"]
    xb = m["precipitation_mm_b"]
    valid = xa.notna() & xb.notna()
    vv_a = xa[valid]
    vv_b = xb[valid]
    precip_rows.append(
        {
            "station_a": a,
            "station_b": b,
            "aligned_hours": int(valid.sum()),
            "mae": calc_mae(vv_a, vv_b),
            "rmse": calc_rmse(vv_a, vv_b),
            "bias": calc_bias(vv_a, vv_b),
            "pearson_r": calc_corr(vv_a, vv_b, "pearson"),
            "spearman_r": calc_corr(vv_a, vv_b, "spearman"),
        }
    )

    # Seasonal similarity can reveal pairs that diverge only in specific seasons.
    for season in ["winter", "spring", "summer", "fall"]:
        sm = m[(m["season_a"] == season) & (m["season_b"] == season)]
        for var in CORE_VARS:
            xa = sm[f"{var}_a"]
            xb = sm[f"{var}_b"]
            valid = xa.notna() & xb.notna()
            vv_a = xa[valid]
            vv_b = xb[valid]
            seasonal_rows.append(
                {
                    "station_a": a,
                    "station_b": b,
                    "season": season,
                    "variable": var,
                    "aligned_hours": int(valid.sum()),
                    "pearson_r": calc_corr(vv_a, vv_b, "pearson"),
                    "rmse": calc_rmse(vv_a, vv_b),
                }
            )

    # Wind U/V comparison uses vector decomposition to respect circular direction behavior.
    u_a = m["wind_speed_kmh_a"] * np.cos(np.deg2rad(m["wind_direction_deg_a"]))
    v_a = m["wind_speed_kmh_a"] * np.sin(np.deg2rad(m["wind_direction_deg_a"]))
    u_b = m["wind_speed_kmh_b"] * np.cos(np.deg2rad(m["wind_direction_deg_b"]))
    v_b = m["wind_speed_kmh_b"] * np.sin(np.deg2rad(m["wind_direction_deg_b"]))
    valid_u = u_a.notna() & u_b.notna()
    valid_v = v_a.notna() & v_b.notna()
    wind_uv_rows.append(
        {
            "station_a": a,
            "station_b": b,
            "aligned_hours_u": int(valid_u.sum()),
            "aligned_hours_v": int(valid_v.sum()),
            "u_corr": calc_corr(u_a[valid_u], u_b[valid_u], "pearson"),
            "v_corr": calc_corr(v_a[valid_v], v_b[valid_v], "pearson"),
        }
    )

core_metrics_df = pd.DataFrame(core_rows)
precip_metrics_df = pd.DataFrame(precip_rows)
seasonal_metrics_df = pd.DataFrame(seasonal_rows)
wind_uv_metrics_df = pd.DataFrame(wind_uv_rows)

core_metrics_df.to_csv(TABLES_DIR / "03_explore_pairwise_core_metrics.csv", index=False)
precip_metrics_df.to_csv(TABLES_DIR / "03_explore_pairwise_precip_metrics.csv", index=False)
seasonal_metrics_df.to_csv(TABLES_DIR / "03_explore_pairwise_seasonal_metrics.csv", index=False)
wind_uv_metrics_df.to_csv(TABLES_DIR / "03_explore_pairwise_wind_uv_metrics.csv", index=False)

# Figure 1: station-variable missingness heatmap
heat = coverage_overall.pivot(index="station_slug", columns="variable", values="missing_pct")
plt.figure(figsize=(10, 5))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Missingness (%) by Station and Variable")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_explore_missingness_heatmap.png", dpi=180)
plt.close()

# Figure 2: monthly missingness trend for core variables
tmp = coverage_monthly[coverage_monthly["variable"].isin(CORE_VARS)].copy()
tmp["month_dt"] = pd.to_datetime(tmp["year_month"] + "-01", utc=True, errors="coerce")
plt.figure(figsize=(12, 6))
sns.lineplot(data=tmp, x="month_dt", y="missing_pct", hue="station_slug", style="variable")
plt.title("Monthly Missingness (%) for Core Variables")
plt.xlabel("Month")
plt.ylabel("Missing %")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_explore_missingness_timeline.png", dpi=180)
plt.close()

display(Markdown("### Station Similarity Snapshot (Business Impact)"))
display(Markdown("Higher correlations and lower RMSE suggest stations are more interchangeable. Lower agreement indicates each station may capture unique local conditions."))

core_similarity = (
    core_metrics_df.groupby(["station_a", "station_b"], dropna=False)
    .agg(avg_pearson_r=("pearson_r", "mean"), avg_rmse=("rmse", "mean"), aligned_hours=("aligned_hours", "min"))
    .reset_index()
    .sort_values(["avg_pearson_r", "avg_rmse"], ascending=[False, True])
)
display(core_similarity.head(8).round(3))

if not core_similarity.empty:
    strongest = core_similarity.iloc[0]
    weakest = core_similarity.iloc[-1]
    display(Markdown(
        f"**Most Similar Pair:** {strongest['station_a']} and {strongest['station_b']} (Avg Pearson r = {strongest['avg_pearson_r']:.3f})."
    ))
    display(Markdown(
        f"**Least Similar Pair:** {weakest['station_a']} and {weakest['station_b']} (Avg Pearson r = {weakest['avg_pearson_r']:.3f}). "
        "This pair may represent distinct micro-climate behavior and should be reviewed before any station consolidation."
    ))

print("Saved EDA tables and figures.")

### Station Similarity Snapshot (Business Impact)

Higher correlations and lower RMSE suggest stations are more interchangeable. Lower agreement indicates each station may capture unique local conditions.

,station_a,station_b,avg_pearson_r,avg_rmse,aligned_hours
13,Stanhope,Tracadie_Wharf,0.941,2.862,0
4,Cavendish,Tracadie_Wharf,0.923,3.807,0
3,Cavendish,Stanley_Bridge_Wharf,0.921,3.440,0
2,Cavendish,Stanhope,0.904,4.219,24532
12,Stanhope,Stanley_Bridge_Wharf,0.884,4.044,0
8,Greenwich,Tracadie_Wharf,0.882,5.417,0
6,Greenwich,Stanhope,0.881,5.665,18338
14,Stanley_Bridge_Wharf,Tracadie_Wharf,0.880,4.735,0


**Most Similar Pair:** Stanhope and Tracadie_Wharf (Avg Pearson r = 0.941).

**Least Similar Pair:** North_Rustico_Wharf and Stanley_Bridge_Wharf (Avg Pearson r = 0.757). This pair may represent distinct micro-climate behavior and should be reviewed before any station consolidation.

Saved EDA tables and figures.


**Expected Output (Cell 7)**
- Saves pairwise similarity CSVs and EDA figures in `outputs/tables` and `outputs/figures`.
- Displays top similarity comparisons and business interpretation text.
- Prints `Saved EDA tables and figures.` when done.

In [13]:
# =============================================================================
# UNALIGNED HOURS PER YEAR (ADHOC)
# Purpose: find unaligned hours by year for Stanhope vs Cavendish/Greenwich.
# =============================================================================

import pandas as pd

# Start with the main hourly cleaning block
df = hourly.copy()
df['year'] = df['datetime_utc'].dt.year

# Look at core comparison: Stanhope vs Cavendish
stan_cav_combined = df[df['station_slug'].isin(['Stanhope', 'Cavendish'])].copy()
stan_cav_wide = stan_cav_combined.pivot(index=['datetime_utc', 'year'], columns='station_slug', values='air_temperature_c')
stan_cav_wide['aligned'] = stan_cav_wide['Cavendish'].notna() & stan_cav_wide['Stanhope'].notna()

print("Cavendish-Stanhope Alignments per Year")
print(stan_cav_wide.groupby('year')['aligned'].value_counts())

# Now Greenwich
stan_grn_combined = df[df['station_slug'].isin(['Stanhope', 'Greenwich'])].copy()
stan_grn_wide = stan_grn_combined.pivot(index=['datetime_utc', 'year'], columns='station_slug', values='air_temperature_c')
stan_grn_wide['aligned'] = stan_grn_wide['Greenwich'].notna() & stan_grn_wide['Stanhope'].notna()

print("\nGreenwich-Stanhope Alignments per Year")
print(stan_grn_wide.groupby('year')['aligned'].value_counts())

Cavendish-Stanhope Alignments per Year
year  aligned
2022  False      7759
      True        997
2023  True       8029
      False       731
2024  True       8773
      False        11
2025  True       6904
      False      1856
2026  False         4
Name: count, dtype: int64

Greenwich-Stanhope Alignments per Year
year  aligned
2022  False      7166
      True       1590
2023  False      6326
      True       2434
2024  True       5553
      False      3231
2025  True       8757
      False         3
2026  True          4
Name: count, dtype: int64


**Expected Output (Cell 8)**
- Prints unaligned and aligned hours specifically broken down per year.
- Computes this comparison for the Stanhope vs Cavendish pair, and Stanhope vs Greenwich pair.

In [14]:
# =============================================================================
# FWI MODELING + VALIDATION
# Purpose: compute daily FWI from hourly weather and validate vs ECCC references.
# =============================================================================

SEASON_MONTHS = {6, 7, 8, 9}
FFMC_START = 85.0
DMC_START = 6.0
DC_START = 15.0


def ffmc_code(temp_c: float, rh_pct: float, wind_kmh: float, rain_mm: float, ffmc_prev: float) -> float:
    """Compute Fine Fuel Moisture Code (FFMC) for one day using standard equations."""
    mo = 147.2 * (101 - ffmc_prev) / (59.5 + ffmc_prev)
    if rain_mm > 0.5:
        rf = rain_mm - 0.5
        if mo > 150:
            mo = mo + 42.5 * rf * math.exp(-100 / (251 - mo)) * (1 - math.exp(-6.93 / rf)) + 0.0015 * (mo - 150) ** 2 * rf ** 0.5
        else:
            mo = mo + 42.5 * rf * math.exp(-100 / (251 - mo)) * (1 - math.exp(-6.93 / rf))
        mo = min(mo, 250)
    ed = 0.942 * rh_pct**0.679 + 11 * math.exp((rh_pct - 100) / 10) + 0.18 * (21.1 - temp_c) * (1 - math.exp(-0.115 * rh_pct))
    ew = 0.618 * rh_pct**0.753 + 10 * math.exp((rh_pct - 100) / 10) + 0.18 * (21.1 - temp_c) * (1 - math.exp(-0.115 * rh_pct))
    if mo > ed:
        ko = 0.424 * (1 - (rh_pct / 100) ** 1.7) + 0.0694 * wind_kmh**0.5 * (1 - (rh_pct / 100) ** 8)
        kd = ko * 0.581 * math.exp(0.0365 * temp_c)
        m = ed + (mo - ed) * 10 ** (-kd)
    elif mo < ew:
        ko = 0.424 * (1 - ((100 - rh_pct) / 100) ** 1.7) + 0.0694 * wind_kmh**0.5 * (1 - ((100 - rh_pct) / 100) ** 8)
        kw = ko * 0.581 * math.exp(0.0365 * temp_c)
        m = ew - (ew - mo) * 10 ** (-kw)
    else:
        m = mo
    return 59.5 * (250 - m) / (147.2 + m)


def dmc_code(temp_c: float, rh_pct: float, rain_mm: float, dmc_prev: float, month: int) -> float:
    """Compute Duff Moisture Code (DMC) for one day."""
    day_length_factors = [6.5, 7.5, 9.0, 12.8, 13.9, 13.9, 12.4, 10.9, 9.4, 8.0, 7.8, 6.3]
    temp_c = max(temp_c, -1.1)
    rk = 1.894 * (temp_c + 1.1) * (100 - rh_pct) * day_length_factors[month - 1] * 1e-4
    if rain_mm > 1.5:
        rw = 0.92 * rain_mm - 1.27
        wmi = 20 + 280 / math.exp(0.023 * dmc_prev)
        if dmc_prev <= 33:
            b = 100 / (0.5 + 0.3 * dmc_prev)
        elif dmc_prev <= 65:
            b = 14 - 1.3 * math.log(dmc_prev)
        else:
            b = 6.2 * math.log(dmc_prev) - 17.2
        wmr = wmi + 1000 * rw / (48.77 + b * rw) if rw > 0 else wmi
        pr = 43.43 * (5.6348 - math.log(wmr - 20)) if wmr > 20 else 0.0
    else:
        pr = dmc_prev
    return max(pr, 0) + rk


def dc_code(temp_c: float, rain_mm: float, dc_prev: float, month: int) -> float:
    """Compute Drought Code (DC) for one day."""
    seasonal_factors = [-1.6, -1.6, -1.6, 0.9, 3.8, 5.8, 6.4, 5.0, 2.4, 0.4, -1.6, -1.6]
    temp_c = max(temp_c, -2.8)
    pe = max((0.36 * (temp_c + 2.8) + seasonal_factors[month - 1]) / 2, 0)
    if rain_mm > 2.8:
        rw = 0.83 * rain_mm - 1.27
        smi = 800 * math.exp(-dc_prev / 400)
        log_arg = 1 + 3.937 * rw / smi if smi > 0 else 1
        dr = dc_prev - 400 * math.log(log_arg) if log_arg > 0 else 0
        dr = max(dr, 0)
    else:
        dr = dc_prev
    return dr + pe


def isi_index(wind_kmh: float, ffmc_value: float) -> float:
    """Compute Initial Spread Index (ISI) from wind and FFMC."""
    fm = 147.2 * (101 - ffmc_value) / (59.5 + ffmc_value)
    sf = 19.115 * math.exp(-0.1386 * fm) * (1 + fm**5.31 / 4.93e7)
    return sf * math.exp(0.05039 * wind_kmh)


def bui_index(dmc_value: float, dc_value: float) -> float:
    """Compute Build-Up Index (BUI) from DMC and DC."""
    den = dmc_value + 0.4 * dc_value
    if den <= 0:
        return 0.0
    if dmc_value <= 0.4 * dc_value:
        return 0.8 * dmc_value * dc_value / den
    return max(dmc_value - (1 - 0.8 * dc_value / den) * (0.92 + (0.0114 * dmc_value) ** 1.7), 0)


def fwi_index(isi_value: float, bui_value: float) -> float:
    """Compute final Fire Weather Index (FWI) from ISI and BUI."""
    if bui_value <= 80:
        bb = 0.1 * isi_value * (0.626 * bui_value**0.809 + 2)
    else:
        bb = 0.1 * isi_value * (1000 / (25 + 108.64 * math.exp(-0.023 * bui_value)))
    return bb if bb <= 1 else math.exp(2.72 * (0.434 * math.log(bb)) ** 0.647)


# ---- Build local-time daily modeling frame from hourly inputs ----
s = hourly.copy()
s["datetime_local"] = s["datetime_utc"].dt.tz_convert(HALIFAX_TZ)
s["date_local"] = s["datetime_local"].dt.date
s["local_hour"] = s["datetime_local"].dt.hour
s = s[s["datetime_local"].dt.month.isin(SEASON_MONTHS)].copy()

# Extract noon observations used by FWI equations.
noon = s[s["local_hour"] == 12][["station_slug", "date_local", "datetime_local", "datetime_utc", *CORE_VARS]].copy()
noon = noon.rename(columns={
    "datetime_local": "datetime_noon_local",
    "datetime_utc": "datetime_noon_utc",
    "air_temperature_c": "air_temperature_c_noon",
    "relative_humidity_pct": "relative_humidity_pct_noon",
    "wind_speed_kmh": "wind_speed_kmh_noon",
})
noon["noon_core_present"] = noon[["air_temperature_c_noon", "relative_humidity_pct_noon", "wind_speed_kmh_noon"]].notna().all(axis=1)

# Compute rolling 24-hour precipitation total aligned to each local noon day.
precip_rows = []
for station_slug, g in s.groupby("station_slug", dropna=False):
    precip = g.set_index("datetime_utc")["precipitation_mm"].sort_index()
    for local_day in sorted(g["date_local"].unique().tolist()):
        start_local = datetime.combine(local_day - timedelta(days=1), datetime.min.time().replace(hour=13), tzinfo=HALIFAX_TZ)
        idx_local = pd.date_range(start=start_local, periods=24, freq="h", tz=HALIFAX_TZ)
        vals = precip.reindex(idx_local.tz_convert(timezone.utc))
        complete = int(vals.notna().sum()) == 24
        precip_rows.append(
            {
                "station_slug": station_slug,
                "date_local": local_day,
                "precip_window_complete_24h": complete,
                "precip_24h_sum_mm": float(vals.sum()) if complete else np.nan,
            }
        )
precip_daily = pd.DataFrame(precip_rows)

# Merge noon weather + precip window completeness into one daily modeling table.
daily = noon.merge(precip_daily, on=["station_slug", "date_local"], how="left")
daily = daily.sort_values(["station_slug", "date_local"]).reset_index(drop=True)

# Pre-allocate model output columns.
daily["ffmc"] = np.nan
daily["dmc"] = np.nan
daily["dc"] = np.nan
daily["isi"] = np.nan
daily["bui"] = np.nan
daily["fwi"] = np.nan
daily["segment_id"] = pd.Series(pd.NA, index=daily.index, dtype="string")
daily["continuity_gap_days"] = np.nan
daily["continuity_reset_applied"] = False

# ---- Sequential FWI calculation by station ----
# Important: FFMC/DMC/DC are stateful, so we iterate in date order.
for station_slug, idx in daily.groupby("station_slug", sort=False).groups.items():
    prev_day = None
    ffmc_prev, dmc_prev, dc_prev = FFMC_START, DMC_START, DC_START
    seg = 0
    for i in idx:
        row = daily.loc[i]
        # Skip days lacking core noon values or complete precip window.
        if not bool(row["noon_core_present"]) or not bool(row["precip_window_complete_24h"]):
            prev_day = None
            continue
        day = row["date_local"]
        gap = float((day - prev_day).days) if prev_day is not None else np.nan

        # Reset moisture-code state when continuity breaks (>1 day gap).
        if prev_day is None or (pd.notna(gap) and gap > 1):
            seg += 1
            ffmc_prev, dmc_prev, dc_prev = FFMC_START, DMC_START, DC_START
            reset = True
        else:
            reset = False

        # Input values for this day.
        t = float(row["air_temperature_c_noon"])
        rh = float(row["relative_humidity_pct_noon"])
        w = float(row["wind_speed_kmh_noon"])
        r = float(row["precip_24h_sum_mm"])
        month = int(pd.Timestamp(day).month)

        # Run one-step update across the FWI code chain.
        ffmc_prev = ffmc_code(t, rh, w, r, ffmc_prev)
        dmc_prev = dmc_code(t, rh, r, dmc_prev, month)
        dc_prev = dc_code(t, r, dc_prev, month)
        isi = isi_index(w, ffmc_prev)
        bui = bui_index(dmc_prev, dc_prev)
        fwi = fwi_index(isi, bui)

        # Persist results and continuity metadata.
        daily.loc[i, ["ffmc", "dmc", "dc", "isi", "bui", "fwi"]] = [ffmc_prev, dmc_prev, dc_prev, isi, bui, fwi]
        daily.loc[i, "segment_id"] = f"{pd.Timestamp(day).year}_{seg}"
        daily.loc[i, "continuity_gap_days"] = gap
        daily.loc[i, "continuity_reset_applied"] = reset
        prev_day = day

# Final modeled daily table export.
model_daily = daily[[
    "station_slug", "date_local", "air_temperature_c_noon", "relative_humidity_pct_noon", "wind_speed_kmh_noon", "precip_24h_sum_mm",
    "ffmc", "dmc", "dc", "isi", "bui", "fwi", "noon_core_present", "precip_window_complete_24h", "segment_id", "continuity_gap_days", "continuity_reset_applied"
]].copy()
model_daily.to_csv(TABLES_DIR / "04_model_fwi_daily.csv", index=False)

# ---- Optional validation against ECCC Stanhope daily FWI references ----
validation_rows = []
if REFERENCE_FWI_DIR.exists() and any(REFERENCE_FWI_DIR.glob("*.csv")):
    ref_frames = []
    for fp in sorted(REFERENCE_FWI_DIR.glob("*.csv")):
        df_ref = pd.read_csv(fp, low_memory=False)
        if all(c in df_ref.columns for c in ["Date", "FFMC", "DMC", "DC", "FWI"]):
            df_ref = df_ref[["Date", "FFMC", "DMC", "DC", "FWI"]].copy()
            df_ref["date_local"] = pd.to_datetime(df_ref["Date"], errors="coerce").dt.date
            ref_frames.append(df_ref.dropna(subset=["date_local"]))
    if ref_frames:
        ref = pd.concat(ref_frames, ignore_index=True).drop_duplicates(subset=["date_local"], keep="last")
        stanhope = model_daily[model_daily["station_slug"].astype(str).str.lower() == REFERENCE_STATION.lower()].copy()
        merged = stanhope.merge(ref, on="date_local", how="inner")
        for pred_col, ref_col in [("ffmc", "FFMC"), ("dmc", "DMC"), ("dc", "DC"), ("fwi", "FWI")]:
            valid = merged[pred_col].notna() & merged[ref_col].notna()
            xa = merged.loc[valid, pred_col].astype(float)
            xb = merged.loc[valid, ref_col].astype(float)
            validation_rows.append(
                {
                    "metric": pred_col,
                    "n": int(valid.sum()),
                    "mae": calc_mae(xa, xb),
                    "rmse": calc_rmse(xa, xb),
                    "bias": calc_bias(xa, xb),
                    "pearson_r": calc_corr(xa, xb, "pearson"),
                }
            )

validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(TABLES_DIR / "04_model_fwi_validation_summary.csv", index=False)

display(Markdown("### Fire Weather Index Output (Business Impact)"))
display(Markdown("These are the daily fire danger indicators used to inform readiness, staffing, and prevention messaging. Better agreement with ECCC means higher confidence in local operational use."))

fwi_snapshot = (
    model_daily.dropna(subset=["fwi"])
    .sort_values("date_local", ascending=False)
    [["station_slug", "date_local", "fwi", "ffmc", "dmc", "dc"]]
    .head(12)
    .rename(columns={
        "station_slug": "Station",
        "date_local": "Date",
        "fwi": "FWI",
        "ffmc": "FFMC",
        "dmc": "DMC",
        "dc": "DC",
    })
)
display(fwi_snapshot.round(2))

if not validation_df.empty:
    validation_view = validation_df.rename(columns={
        "metric": "Metric",
        "n": "Days Compared",
        "mae": "MAE",
        "rmse": "RMSE",
        "bias": "Bias",
        "pearson_r": "Pearson r",
    })
    display(Markdown("**Validation Against ECCC Stanhope**"))
    display(validation_view.round(3))
    fwi_row = validation_df[validation_df["metric"] == "fwi"]
    if not fwi_row.empty:
        r_val = float(fwi_row.iloc[0]["pearson_r"])
        rmse_val = float(fwi_row.iloc[0]["rmse"])
        confidence = "high" if r_val >= 0.85 else ("moderate" if r_val >= 0.70 else "low")
        display(Markdown(
            f"**Operational Confidence Signal:** FWI correlation to ECCC is {r_val:.3f} with RMSE {rmse_val:.3f}. "
            f"This suggests **{confidence}** confidence for using this pipeline in day-to-day fire danger planning."
        ))

print("Saved FWI model and validation outputs.")

### Fire Weather Index Output (Business Impact)

These are the daily fire danger indicators used to inform readiness, staffing, and prevention messaging. Better agreement with ECCC means higher confidence in local operational use.

,Station,Date,FWI,FFMC,DMC,DC
1585,Stanhope,2025-09-30,4.63,85.16,7.85,24.99
365,Cavendish,2025-09-30,11.41,82.88,40.85,586.38
731,Greenwich,2025-09-30,7.34,80.30,41.60,641.61
1097,North_Rustico_Wharf,2025-09-30,13.96,84.69,42.30,576.07
1096,North_Rustico_Wharf,2025-09-29,18.35,82.56,40.97,571.42
364,Cavendish,2025-09-29,7.61,77.87,39.62,581.82
730,Greenwich,2025-09-29,4.15,70.23,40.27,636.86
1584,Stanhope,2025-09-29,2.32,80.68,6.28,20.24
729,Greenwich,2025-09-28,10.40,78.98,50.96,646.64
363,Cavendish,2025-09-28,11.25,80.49,44.18,576.47


**Validation Against ECCC Stanhope**

,Metric,Days Compared,MAE,RMSE,Bias,Pearson r
0,ffmc,467,2.461,4.988,-0.309,0.968
1,dmc,467,4.677,7.989,-3.852,0.918
2,dc,467,88.901,141.399,-88.877,0.645
3,fwi,467,1.914,3.249,-1.631,0.936


**Operational Confidence Signal:** FWI correlation to ECCC is 0.936 with RMSE 3.249. This suggests **high** confidence for using this pipeline in day-to-day fire danger planning.

Saved FWI model and validation outputs.


**Expected Output (Cell 9)**
- Saves daily modeled FWI results and validation summary CSVs to `outputs/tables`.
- Displays recent FWI snapshot and, when available, validation metrics vs ECCC Stanhope.
- Prints `Saved FWI model and validation outputs.` at completion.

In [15]:
# =============================================================================
# REDUNDANCY MODELING: PCA + K-MEANS + BENCHMARKING
# Purpose: quantify overlap and uniqueness of station signals.
# =============================================================================

# Use complete-case rows for stable multivariate modeling.
pca_data = hourly[["datetime_utc", "station_slug", "air_temperature_c", "relative_humidity_pct", "wind_speed_kmh", "precipitation_mm"]].dropna().copy()
features = ["air_temperature_c", "relative_humidity_pct", "wind_speed_kmh", "precipitation_mm"]

if len(pca_data) < 10:
    fail("Insufficient complete-case rows for PCA/clustering.")

# Standardize features so each variable contributes comparably.
X = pca_data[features].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---- PCA workflow ----
n_comp = min(4, len(features))
pca = PCA(n_components=n_comp, random_state=RANDOM_SEED)
scores = pca.fit_transform(X_scaled)

explained = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(n_comp)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "explained_variance_cumulative": np.cumsum(pca.explained_variance_ratio_),
})
explained.to_csv(TABLES_DIR / "05_model_pca_explained_variance.csv", index=False)

# Component loadings show which variables drive each principal component.
loading_rows = []
for i in range(n_comp):
    for var, val in zip(features, pca.components_[i]):
        loading_rows.append({"component": f"PC{i+1}", "variable": var, "loading": float(val)})
loadings = pd.DataFrame(loading_rows)
loadings.to_csv(TABLES_DIR / "05_model_pca_loadings.csv", index=False)

# Long-format PC scores for downstream summaries/plots.
score_df = pca_data[["datetime_utc", "station_slug"]].copy()
for i in range(n_comp):
    score_df[f"PC{i+1}"] = scores[:, i]
score_long = score_df.melt(id_vars=["datetime_utc", "station_slug"], var_name="pc", value_name="score")
score_long.to_csv(TABLES_DIR / "05_model_pca_scores.csv", index=False)

station_summary = score_long.groupby(["station_slug", "pc"], dropna=False)["score"].agg(["count", "mean", "std", "median"]).reset_index()
station_summary = station_summary.rename(columns={"count": "n_rows"})
station_summary.to_csv(TABLES_DIR / "05_model_pca_station_summary.csv", index=False)

# ---- Pairwise benchmarking vs Stanhope reference ----
ref_slug = REFERENCE_STATION
bench_rows = []
for stn in sorted(hourly["station_slug"].dropna().unique().tolist()):
    if str(stn).lower() == ref_slug.lower():
        continue
    da = hourly[hourly["station_slug"] == stn][["datetime_utc", *features]].rename(columns={c: f"{c}_a" for c in features})
    db = hourly[hourly["station_slug"].astype(str).str.lower() == ref_slug.lower()][["datetime_utc", *features]].rename(columns={c: f"{c}_b" for c in features})
    m = da.merge(db, on="datetime_utc", how="inner")
    for var in features:
        xa = m[f"{var}_a"]
        xb = m[f"{var}_b"]
        valid = xa.notna() & xb.notna()
        va = xa[valid].astype(float)
        vb = xb[valid].astype(float)
        bench_rows.extend(
            [
                {"station_slug": stn, "reference_station_slug": ref_slug, "variable": var, "metric": "mae", "value": calc_mae(va, vb), "aligned_hours": int(valid.sum())},
                {"station_slug": stn, "reference_station_slug": ref_slug, "variable": var, "metric": "rmse", "value": calc_rmse(va, vb), "aligned_hours": int(valid.sum())},
                {"station_slug": stn, "reference_station_slug": ref_slug, "variable": var, "metric": "bias", "value": calc_bias(va, vb), "aligned_hours": int(valid.sum())},
                {"station_slug": stn, "reference_station_slug": ref_slug, "variable": var, "metric": "pearson_r", "value": calc_corr(va, vb, "pearson"), "aligned_hours": int(valid.sum())},
                {"station_slug": stn, "reference_station_slug": ref_slug, "variable": var, "metric": "spearman_r", "value": calc_corr(va, vb, "spearman"), "aligned_hours": int(valid.sum())},
            ]
        )
benchmark_df = pd.DataFrame(bench_rows)
benchmark_df.to_csv(TABLES_DIR / "05_model_hourly_benchmarks.csv", index=False)

# ---- PCA biplot ----
# Visual summary of station distributions in PC-space + variable loading vectors.
plt.figure(figsize=(9, 7))
plot_df = score_df.copy()
for stn, g in plot_df.groupby("station_slug", dropna=False):
    plt.scatter(g["PC1"], g["PC2"] if "PC2" in g.columns else np.zeros(len(g)), s=8, alpha=0.35, label=stn)

for _, row in loadings[loadings["component"].isin(["PC1", "PC2"])].pivot(index="variable", columns="component", values="loading").reset_index().iterrows():
    x = float(row.get("PC1", 0.0))
    y = float(row.get("PC2", 0.0))
    plt.arrow(0, 0, x * 4, y * 4, color="black", alpha=0.7, head_width=0.03)
    plt.text(x * 4.2, y * 4.2, row["variable"], fontsize=9)

plt.title("PCA Biplot (PC1 vs PC2)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_model_pca_biplot.png", dpi=180)
plt.close()

# ---- K-Means workflow ----
kmeans_summary = []
if DO_KMEANS:
    for k in K_VALUES:
        if k >= 2 and k <= len(pca_data):
            km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
            labels = km.fit_predict(X_scaled)
            inertia = float(km.inertia_)
            kmeans_summary.append({"k": k, "inertia": inertia})
    if kmeans_summary:
        pd.DataFrame(kmeans_summary).to_csv(TABLES_DIR / "05_model_kmeans_elbow.csv", index=False)
        # Current strategy picks the lowest inertia among tested k values.
        k_best = sorted(kmeans_summary, key=lambda x: x["inertia"])[0]["k"]
        km = KMeans(n_clusters=int(k_best), random_state=RANDOM_SEED, n_init=10)
        labels = km.fit_predict(X_scaled)
        clusters = pca_data[["datetime_utc", "station_slug"]].copy()
        clusters["cluster"] = labels
        clusters.to_csv(TABLES_DIR / "05_model_kmeans_assignments.csv", index=False)

display(Markdown("### Redundancy Modeling Summary (Business Impact)"))
display(Markdown("This section estimates which stations provide unique value versus overlapping signals. Higher overlap suggests potential consolidation opportunities; low overlap indicates higher risk in removing stations."))

display(Markdown("**PCA Explained Variance**"))
display(explained.round(4))

benchmark_rollup = (
    benchmark_df[benchmark_df["metric"].isin(["pearson_r", "rmse"])]
    .pivot_table(
        index=["station_slug"],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
    .rename(columns={
        "station_slug": "Station",
        "pearson_r": "Avg Correlation vs Stanhope",
        "rmse": "Avg RMSE vs Stanhope",
    })
    .sort_values(["Avg Correlation vs Stanhope", "Avg RMSE vs Stanhope"], ascending=[False, True])
)
display(Markdown("**Station Similarity to Stanhope (Higher correlation + lower RMSE = more redundant)**"))
display(benchmark_rollup.round(3))

if not benchmark_rollup.empty:
    best = benchmark_rollup.iloc[0]
    worst = benchmark_rollup.iloc[-1]
    display(Markdown(
        f"**Most Redundant Candidate Signal:** {best['Station']} appears closest to Stanhope in this run. "
        "This may indicate lower information loss if removed, pending policy and local context review."
    ))
    display(Markdown(
        f"**Most Unique Candidate Signal:** {worst['Station']} appears least similar to Stanhope. "
        "This station may hold distinct micro-climate value and should be protected in consolidation scenarios."
    ))

print("Saved PCA/clustering and benchmarking outputs.")

### Redundancy Modeling Summary (Business Impact)

This section estimates which stations provide unique value versus overlapping signals. Higher overlap suggests potential consolidation opportunities; low overlap indicates higher risk in removing stations.

**PCA Explained Variance**

,component,explained_variance_ratio,explained_variance_cumulative
0,PC1,0.3138,0.3138
1,PC2,0.2756,0.5894
2,PC3,0.2467,0.8361
3,PC4,0.1639,1.0000


**Station Similarity to Stanhope (Higher correlation + lower RMSE = more redundant)**

metric,Station,Avg Correlation vs Stanhope,Avg RMSE vs Stanhope
4,Tracadie_Wharf,0.795,2.083
0,Cavendish,0.785,3.313
1,Greenwich,0.784,4.386
3,Stanley_Bridge_Wharf,0.728,2.879
2,North_Rustico_Wharf,0.723,4.077


**Most Redundant Candidate Signal:** Tracadie_Wharf appears closest to Stanhope in this run. This may indicate lower information loss if removed, pending policy and local context review.

**Most Unique Candidate Signal:** North_Rustico_Wharf appears least similar to Stanhope. This station may hold distinct micro-climate value and should be protected in consolidation scenarios.

Saved PCA/clustering and benchmarking outputs.


**Expected Output (Cell 10)**
- Saves PCA, benchmark, and clustering artifacts to `outputs/tables` and a PCA biplot to `outputs/figures`.
- Displays PCA variance table and station redundancy comparison against Stanhope.
- Prints `Saved PCA/clustering and benchmarking outputs.` when finished.

In [ ]:
# =============================================================================
# UNCERTAINTY + DECISION LAYER + REPORT EXPORT
# Purpose: convert model outputs into removal-risk probabilities and clear decisions.
# =============================================================================

# ---- Correlation diagnostics vs Stanhope (hourly true variables) ----
corr_rows = []
ref_df = hourly[hourly["station_slug"].astype(str).str.lower() == REFERENCE_STATION.lower()][["datetime_utc", *CORE_VARS, "precipitation_mm"]]
for stn in CANDIDATE_STATIONS:
    cand = hourly[hourly["station_slug"].astype(str).str.lower() == stn.lower()][["datetime_utc", *CORE_VARS, "precipitation_mm"]]
    m = cand.merge(ref_df, on="datetime_utc", how="inner", suffixes=("_cand", "_ref"))
    for var in [*CORE_VARS, "precipitation_mm"]:
        v1 = m[f"{var}_cand"]
        v2 = m[f"{var}_ref"]
        valid = v1.notna() & v2.notna()
        corr_rows.append(
            {
                "candidate_station": stn,
                "reference_station": REFERENCE_STATION,
                "variable": var,
                "aligned_hours": int(valid.sum()),
                "pearson_r": calc_corr(v1[valid], v2[valid], "pearson"),
                "spearman_r": calc_corr(v1[valid], v2[valid], "spearman"),
            }
        )

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(TABLES_DIR / "06_interpret_fwi_true_stanhope_correlations.csv", index=False)

# ---- Redundancy risk from daily FWI deltas ----
# Logic: if candidate FWI often differs materially from Stanhope, removal risk is higher.
risk_rows = []
stan = model_daily[model_daily["station_slug"].astype(str).str.lower() == REFERENCE_STATION.lower()][["date_local", "fwi"]].rename(columns={"fwi": "fwi_ref"})
for stn in CANDIDATE_STATIONS:
    cand = model_daily[model_daily["station_slug"].astype(str).str.lower() == stn.lower()][["date_local", "fwi"]].rename(columns={"fwi": "fwi_cand"})
    m = cand.merge(stan, on="date_local", how="inner")
    m = m.dropna(subset=["fwi_cand", "fwi_ref"])
    deltas = (m["fwi_cand"] - m["fwi_ref"]).to_numpy(dtype=float)
    abs_deltas = np.abs(deltas)
    n = len(deltas)

    if n == 0:
        risk_rows.append({"candidate_station": stn, "aligned_days": 0, "mean_delta": np.nan, "ci_low": np.nan, "ci_high": np.nan, "prob_abs_delta_gt_threshold": np.nan, "threshold": FWI_DELTA_THRESHOLD})
        continue

    # Bootstrap 95% confidence interval for mean FWI delta.
    rng = np.random.default_rng(RANDOM_SEED)
    draws = []
    for _ in range(1000):
        sample = rng.choice(deltas, size=n, replace=True)
        draws.append(float(np.mean(sample)))
    ci_low = float(np.quantile(draws, 0.025))
    ci_high = float(np.quantile(draws, 0.975))

    # Estimate P(|delta| > threshold).
    # Start with empirical frequency; optionally refine with KDE if scipy is available.
    prob = float(np.mean(abs_deltas > FWI_DELTA_THRESHOLD))
    try:
        from scipy.stats import gaussian_kde  # type: ignore

        kde = gaussian_kde(abs_deltas)
        grid = np.linspace(0, max(abs_deltas.max(), FWI_DELTA_THRESHOLD * 3), 500)
        dens = kde(grid)
        area = np.trapz(dens, grid)
        tail_mask = grid >= FWI_DELTA_THRESHOLD
        tail_area = np.trapz(dens[tail_mask], grid[tail_mask])
        if area > 0:
            prob = float(tail_area / area)
    except Exception:
        pass

    risk_rows.append(
        {
            "candidate_station": stn,
            "aligned_days": n,
            "mean_delta": float(np.mean(deltas)),
            "ci_low": ci_low,
            "ci_high": ci_high,
            "prob_abs_delta_gt_threshold": prob,
            "threshold": FWI_DELTA_THRESHOLD,
            "recommendation_risk_label": "higher_risk" if prob >= 0.35 else "lower_risk",
        }
    )

risk_df = pd.DataFrame(risk_rows)
risk_df.to_csv(TABLES_DIR / "06_interpret_redundancy_risk.csv", index=False)

# ---- Plain-language decision summary ----
summary_rows = []
for _, row in risk_df.iterrows():
    stn = row["candidate_station"]
    prob = row["prob_abs_delta_gt_threshold"]
    aligned = row["aligned_days"]
    statement = (
        "Removing this station may lose meaningful micro-climate signal."
        if pd.notna(prob) and prob >= 0.35
        else "Station appears more redundant relative to Stanhope under current threshold."
    )
    summary_rows.append(
        {
            "station": stn,
            "aligned_days": aligned,
            "probability_abs_fwi_delta_gt_threshold": prob,
            "threshold": FWI_DELTA_THRESHOLD,
            "plain_language_interpretation": statement,
            "next_step": "Review with domain experts before station removal decisions.",
        }
    )

decision_summary = pd.DataFrame(summary_rows)
decision_summary.to_csv(TABLES_DIR / "06_interpret_decision_summary.csv", index=False)

display(Markdown("### Decision Dashboard (Business Impact)"))
display(Markdown("This is the decision-facing output: probability that removing each candidate station would hide meaningful fire-weather differences."))

risk_view = risk_df.rename(columns={
    "candidate_station": "Station",
    "aligned_days": "Days Compared",
    "mean_delta": "Avg FWI Delta",
    "ci_low": "95% CI Low",
    "ci_high": "95% CI High",
    "prob_abs_delta_gt_threshold": "Prob(|FWI Delta| > Threshold)",
    "recommendation_risk_label": "Risk Label",
})
display(risk_view.round(3))

display(Markdown("**Plain-Language Recommendation Summary**"))
display(decision_summary)

if not decision_summary.empty:
    top_risk = decision_summary.sort_values("probability_abs_fwi_delta_gt_threshold", ascending=False).iloc[0]
    top_prob = top_risk["probability_abs_fwi_delta_gt_threshold"]
    if pd.notna(top_prob):
        display(Markdown(
            f"**Highest Removal Risk:** {top_risk['station']} with probability {float(top_prob):.3f} that meaningful FWI signal is lost if removed."
        ))

display(Markdown("### Artifact Export Confirmation"))
display(Markdown("The same outputs shown above are also saved to disk for downstream use and sharing."))

print("Saved uncertainty outputs.")
print("Generated artifacts:")
for p in sorted(TABLES_DIR.glob("*.csv")):
    print(" -", p)
for p in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", p)

### Decision Dashboard (Business Impact)

This is the decision-facing output: probability that removing each candidate station would hide meaningful fire-weather differences.

,Station,Days Compared,Avg FWI Delta,95% CI Low,95% CI High,Prob(|FWI Delta| > Threshold),threshold,Risk Label
0,Cavendish,319,1.608,1.234,1.965,0.426,2.0,higher_risk
1,Greenwich,233,1.850,1.483,2.288,0.468,2.0,higher_risk


**Plain-Language Recommendation Summary**

,station,aligned_days,probability_abs_fwi_delta_gt_threshold,threshold,plain_language_interpretation,next_step
0,Cavendish,319,0.426332,2.0,Removing this station may lose meaningful micr...,Review with domain experts before station remo...
1,Greenwich,233,0.467811,2.0,Removing this station may lose meaningful micr...,Review with domain experts before station remo...


**Highest Removal Risk:** Greenwich with probability 0.468 that meaningful FWI signal is lost if removed.

### Artifact Export Confirmation

The same outputs shown above are also saved to disk for downstream use and sharing.

Saved uncertainty outputs and static report.
Generated artifacts:
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_coverage_monthly.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_coverage_overall.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_noon_readiness.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_pairwise_core_metrics.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_pairwise_precip_metrics.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_pairwise_seasonal_metrics.csv
 - c:\Users\matte\Documents\Holland College\3210_Advanced_Concepts\Agentic_Coding_PC\outputs\tables\03_explore_pairwise_win

**Expected Output (Cell 11)**
- Saves uncertainty/risk CSV outputs to `outputs/tables`.
- Displays a decision dashboard and plain-language recommendation table.
- Prints a generated artifact list and `Saved uncertainty outputs.`